# Deep learning comparison - MLP vs LightGBM vs Random Forest

The literature says gradient-boosted trees usually beat neural networks on tabular fraud data, especially with few positive examples (778 fraud rows in the training window here). This notebook tests that on this project's data instead of just citing it.

Same fairness rules as the Random Forest comparison in `05_model_enhancement.ipynb`:

- same training window (days 5-7) and same held-out set (days 14-16) as every other model
- the MLP gets its own class weight, found by a sweep - not LightGBM's 20 or Random Forest's 500
- the MLP gets its own cost-optimal REVIEW / BLOCK thresholds, using the same cost function as notebook 05
- PR-AUC with the same bootstrap CI method

This is one reasonable attempt at an MLP, at about the same effort the other two models got (fixed architecture, one weight sweep). It is not an extensive tuning exercise.

One difference from notebook 05: the class weight and early stopping are chosen on eval_1 (days 8-10), not on held-out. A neural network needs a validation set to decide when to stop training, and using held-out for that would leak it into the model.

## Setup

In [1]:
from pathlib import Path
import json
import time
import joblib
import numpy as np
import pandas as pd
import torch
import torch.nn as nn
from sklearn.metrics import average_precision_score, precision_score, recall_score, f1_score

PROJECT_ROOT = Path("..").resolve()
MODEL_DIR = PROJECT_ROOT / "models"
FEATURES_PATH = PROJECT_ROOT / "data" / "processed" / "paysim_features.parquet"
REPORT_DIR = PROJECT_ROOT / "reports"
REPORT_DIR.mkdir(exist_ok=True)
RESULTS_PATH = REPORT_DIR / "07_mlp_results.json"

SEED = 42
EVAL1_DAYS = [8, 9, 10]

# Fixed architecture and optimiser settings - not tuned.
HIDDEN = (64, 32, 16)
DROPOUT = 0.2
LEARNING_RATE = 1e-3
WEIGHT_DECAY = 1e-5
BATCH_SIZE = 4096
MAX_EPOCHS = 30
PATIENCE = 5

torch.set_num_threads(8)
print("torch", torch.__version__)

torch 2.14.1+cpu


## Data
Train and held-out are the exact saved sets every other model used. eval_1 (days 8-10) is rebuilt from the feature parquet the same way `02_modeling.ipynb` does it, and is only used for early stopping and for picking the class weight.

In [2]:
X_train, y_train = joblib.load(MODEL_DIR / "train_data.pkl")
X_held_out, y_held_out = joblib.load(MODEL_DIR / "held_out_data.pkl")
feature_columns = list(X_train.columns)
assert list(X_held_out.columns) == feature_columns

df = pd.read_parquet(FEATURES_PATH, columns=feature_columns + ["isFraud", "day_bucket"])
eval1 = df[df["day_bucket"].isin(EVAL1_DAYS)].copy()
for col in ["destination_transactions_last_24h", "destination_transactions_last_7d"]:
    eval1[col] = eval1[col].fillna(0)   # same fill as 02_modeling.ipynb
X_val, y_val = eval1[feature_columns], eval1["isFraud"]
del df

for name, X, y in [("train", X_train, y_train), ("eval_1 (validation)", X_val, y_val), ("held_out", X_held_out, y_held_out)]:
    print(f"  {name:20s} {len(X):>10,} rows  {int(y.sum()):>5,} fraud  ({y.mean()*100:.4f}%)")

  train                 1,311,225 rows    778 fraud  (0.0593%)
  eval_1 (validation)   1,228,723 rows    799 fraud  (0.0650%)
  held_out              1,202,642 rows    822 fraud  (0.0683%)


## Preprocessing
Trees don't care about feature scale; a neural network does. Amounts and balance errors here span from 0 to 10 million with heavy tails, so the continuous features get a signed log transform and are then standardised using **training** statistics only. Binary and one-hot columns are left as 0/1.

In [3]:
binary_columns = [c for c in feature_columns
                  if c.startswith("type_") or c in ("destination_is_first_transaction", "destination_balance_is_zero")]
continuous_columns = [c for c in feature_columns if c not in binary_columns]

def signed_log(frame):
    values = frame[continuous_columns].to_numpy(dtype=np.float64)
    return np.sign(values) * np.log1p(np.abs(values))

train_log = signed_log(X_train)
mean, std = train_log.mean(axis=0), train_log.std(axis=0)
std[std == 0] = 1.0

def preprocess(frame):
    cont = (signed_log(frame) - mean) / std
    binary = frame[binary_columns].to_numpy(dtype=np.float64)
    return torch.tensor(np.hstack([cont, binary]), dtype=torch.float32)

T_train, T_val, T_held = preprocess(X_train), preprocess(X_val), preprocess(X_held_out)
t_y_train = torch.tensor(y_train.to_numpy(), dtype=torch.float32)
print(f"{len(continuous_columns)} continuous + {len(binary_columns)} binary features -> input size {T_train.shape[1]}")

10 continuous + 7 binary features -> input size 17


## The model
Three hidden layers (64, 32, 16), ReLU, dropout after each. Class imbalance is handled in the loss with `pos_weight` - the same idea as `scale_pos_weight` in LightGBM: each fraud row counts that many times more than a legitimate one. Training stops when eval_1 PR-AUC hasn't improved for 5 epochs, and the best epoch's weights are kept.

In [4]:
def make_mlp(n_inputs):
    layers, size = [], n_inputs
    for hidden in HIDDEN:
        layers += [nn.Linear(size, hidden), nn.ReLU(), nn.Dropout(DROPOUT)]
        size = hidden
    layers.append(nn.Linear(size, 1))
    return nn.Sequential(*layers)


def predict_proba(model, T, batch=262144):
    model.eval()
    with torch.no_grad():
        return torch.cat([torch.sigmoid(model(T[i:i + batch]).squeeze(1)) for i in range(0, len(T), batch)]).numpy()


def train_mlp(pos_weight, seed=SEED, verbose=False):
    torch.manual_seed(seed)
    model = make_mlp(T_train.shape[1])
    optimiser = torch.optim.Adam(model.parameters(), lr=LEARNING_RATE, weight_decay=WEIGHT_DECAY)
    loss_fn = nn.BCEWithLogitsLoss(pos_weight=torch.tensor(float(pos_weight)))
    generator = torch.Generator().manual_seed(seed)

    best_score, best_state, best_epoch, stale = -1.0, None, 0, 0
    for epoch in range(1, MAX_EPOCHS + 1):
        model.train()
        order = torch.randperm(len(T_train), generator=generator)
        for i in range(0, len(order), BATCH_SIZE):
            idx = order[i:i + BATCH_SIZE]
            optimiser.zero_grad()
            loss_fn(model(T_train[idx]).squeeze(1), t_y_train[idx]).backward()
            optimiser.step()
        score = average_precision_score(y_val, predict_proba(model, T_val))
        if verbose:
            print(f"    epoch {epoch:2d}  eval_1 PR-AUC = {score:.4f}")
        if score > best_score:
            best_score, best_epoch, stale = score, epoch, 0
            best_state = {k: v.clone() for k, v in model.state_dict().items()}
        else:
            stale += 1
            if stale >= PATIENCE:
                break
    model.load_state_dict(best_state)
    return model, best_score, best_epoch

## Finding the MLP's own class weight
Same idea as the weight sweeps in notebook 05, scored on eval_1. The held-out column is printed for transparency but is not used to choose.

In [5]:
sweep, mlp_models = [], {}
for w in [1, 5, 20, 50, 100, 500]:
    start = time.time()
    model, val_score, epoch = train_mlp(w)
    mlp_models[w] = model
    held_score = average_precision_score(y_held_out, predict_proba(model, T_held))
    sweep.append({"pos_weight": w, "best_epoch": epoch, "validation_pr_auc": float(val_score), "held_out_pr_auc": float(held_score)})
    print(f"pos_weight={w:>4}: eval_1 PR-AUC = {val_score:.4f} (epoch {epoch:2d})   held_out PR-AUC = {held_score:.4f}   [{time.time() - start:.0f}s]")

best_weight = max(sweep, key=lambda r: r["validation_pr_auc"])["pos_weight"]
print(f"\nChosen on eval_1: pos_weight = {best_weight}")

pos_weight=   1: eval_1 PR-AUC = 0.8271 (epoch 24)   held_out PR-AUC = 0.8486   [101s]


pos_weight=   5: eval_1 PR-AUC = 0.8321 (epoch 23)   held_out PR-AUC = 0.8492   [92s]


pos_weight=  20: eval_1 PR-AUC = 0.8336 (epoch 17)   held_out PR-AUC = 0.8485   [74s]


pos_weight=  50: eval_1 PR-AUC = 0.8323 (epoch 23)   held_out PR-AUC = 0.8477   [92s]


pos_weight= 100: eval_1 PR-AUC = 0.8315 (epoch 22)   held_out PR-AUC = 0.8512   [86s]


pos_weight= 500: eval_1 PR-AUC = 0.8296 (epoch 23)   held_out PR-AUC = 0.8475   [89s]

Chosen on eval_1: pos_weight = 20


## Held-out PR-AUC with bootstrap CI
Same bootstrap as notebook 05 (`bootstrap_pr_auc_fast`: 200 resamples, seed 42). LightGBM and Random Forest are re-scored here with the same code so all three numbers come from one place.

In [6]:
def bootstrap_pr_auc_fast(y_true, probs, n_iterations=200, seed=42):
    rng = np.random.RandomState(seed)
    y_true = np.asarray(y_true)
    probs = np.asarray(probs)
    n = len(y_true)
    scores = []
    for _ in range(n_iterations):
        idx = rng.randint(0, n, n)
        scores.append(average_precision_score(y_true[idx], probs[idx]))
    return np.array(scores)


mlp_model = mlp_models[best_weight]
lgb_model = joblib.load(MODEL_DIR / "fraud_model_full.pkl")
rf_model = joblib.load(MODEL_DIR / "rf_model.pkl")

y_true = y_held_out.to_numpy()
probs = {
    "LightGBM": lgb_model.predict_proba(X_held_out)[:, 1],
    "Random Forest": rf_model.predict_proba(X_held_out)[:, 1],
    "MLP": predict_proba(mlp_model, T_held),
}

pr_auc, ci = {}, {}
for name, p in probs.items():
    pr_auc[name] = float(average_precision_score(y_true, p))
    ci[name] = [float(v) for v in np.percentile(bootstrap_pr_auc_fast(y_true, p), [2.5, 97.5])]
    print(f"{name:14s} held_out PR-AUC = {pr_auc[name]:.4f}   95% CI = [{ci[name][0]:.4f}, {ci[name][1]:.4f}]")

mlp_train_pr_auc = float(average_precision_score(y_train, predict_proba(mlp_model, T_train)))
print(f"\nMLP PR-AUC on TRAINING data: {mlp_train_pr_auc:.4f}  (gap vs held_out = overfitting check)")

LightGBM       held_out PR-AUC = 0.7198   95% CI = [0.6877, 0.7492]


Random Forest  held_out PR-AUC = 0.8335   95% CI = [0.8085, 0.8546]


MLP            held_out PR-AUC = 0.8485   95% CI = [0.8217, 0.8720]



MLP PR-AUC on TRAINING data: 0.8711  (gap vs held_out = overfitting check)


## Seed variance
Unlike the tree models here, MLP training is random (weight initialisation, batch order, dropout). Retraining the chosen configuration with five seeds to see how much the held-out number moves.

In [7]:
seed_scores = [pr_auc["MLP"]]
for s in [1, 2, 3, 4]:
    m, _, _ = train_mlp(best_weight, seed=s)
    seed_scores.append(float(average_precision_score(y_true, predict_proba(m, T_held))))
print("Held-out PR-AUC across seeds [42, 1, 2, 3, 4]:", [f"{s:.4f}" for s in seed_scores])
print(f"mean = {np.mean(seed_scores):.4f}, std = {np.std(seed_scores):.4f}, min = {min(seed_scores):.4f}, max = {max(seed_scores):.4f}")

Held-out PR-AUC across seeds [42, 1, 2, 3, 4]: ['0.8485', '0.8491', '0.8498', '0.8505', '0.8526']
mean = 0.8501, std = 0.0014, min = 0.8485, max = 0.8526


## The MLP's own cost-optimal thresholds
Same cost function as notebook 05 (`total_cost`): a missed fraud costs its full amount, wrongly blocking a legitimate transaction costs 5, and anything sent to REVIEW costs 1. Written with array operations instead of a Python loop, same result.

Same procedure as for Random Forest: sweep the BLOCK threshold, then the REVIEW threshold, then narrow in. As in notebook 05, the thresholds are swept on the held-out set, so the cost figures are in-sample for the thresholds for all three models alike.

In [8]:
amounts = X_held_out["amount"].to_numpy()

def total_cost(y_true, probs, amounts, t_block, t_review=0.0001, cost_false_block=5, cost_review=1):
    block = probs >= t_block
    review = (probs >= t_review) & ~block
    allow = ~block & ~review
    return float(amounts[(y_true == 1) & allow].sum()
                 + cost_false_block * ((y_true == 0) & block).sum()
                 + cost_review * review.sum())

# Sanity check: this should reproduce the notebook 05 figures for the other two models.
print(f"LightGBM      at review=0.0001, block=0.90: total cost = {total_cost(y_true, probs['LightGBM'], amounts, 0.90, 0.0001):,.0f}")
print(f"Random Forest at review=0.32,   block=0.85: total cost = {total_cost(y_true, probs['Random Forest'], amounts, 0.85, 0.32):,.0f}")

p_mlp = probs["MLP"]
print("\nMLP probability distribution on fraud:")
print(pd.Series(p_mlp[y_true == 1]).describe().to_string())
print("\nMLP probability distribution on legitimate:")
print(pd.Series(p_mlp[y_true == 0]).describe().to_string())

LightGBM      at review=0.0001, block=0.90: total cost = 46,455,419
Random Forest at review=0.32,   block=0.85: total cost = 47,547

MLP probability distribution on fraud:
count    822.000000
mean       0.823687
std        0.355773
min        0.000037
25%        0.987914
50%        0.999883
75%        0.999998
max        1.000000

MLP probability distribution on legitimate:
count    1.201820e+06
mean     1.179663e-03
std      1.256756e-02
min      1.056968e-16
25%      3.401966e-11
50%      8.767170e-09
75%      2.531548e-07
max      9.888324e-01


In [9]:
BLOCK_GRID = [0.5, 0.6, 0.7, 0.75, 0.8, 0.85, 0.9, 0.95]

print("Step 1 - MLP BLOCK threshold (t_review = 0.0001, as in notebook 05):")
block_sweep = []
for t in BLOCK_GRID:
    c = total_cost(y_true, p_mlp, amounts, t_block=t)
    block_sweep.append({"t_block": t, "cost": c})
    print(f"  t_block={t}: total cost = {c:,.0f}")
t_block_step1 = min(block_sweep, key=lambda r: r["cost"])["t_block"]

print(f"\nStep 2 - MLP REVIEW threshold (holding t_block = {t_block_step1}):")
review_sweep = []
for t_rev in [0.0001, 0.001, 0.01, 0.05, 0.1, 0.2, 0.3, 0.4, 0.5]:
    if t_rev >= t_block_step1:
        continue
    c = total_cost(y_true, p_mlp, amounts, t_block=t_block_step1, t_review=t_rev)
    n_review_legit = int(((p_mlp >= t_rev) & (p_mlp < t_block_step1) & (y_true == 0)).sum())
    n_missed = int(((p_mlp < t_rev) & (y_true == 1)).sum())
    review_sweep.append({"t_review": t_rev, "cost": c, "legit_to_review": n_review_legit, "fraud_missed": n_missed})
    print(f"  t_review={t_rev}: total cost = {c:,.0f}, legit txns sent to REVIEW = {n_review_legit}, fraud missed entirely = {n_missed}")

Step 1 - MLP BLOCK threshold (t_review = 0.0001, as in notebook 05):
  t_block=0.5: total cost = 54,738
  t_block=0.6: total cost = 54,614
  t_block=0.7: total cost = 54,495
  t_block=0.75: total cost = 54,450
  t_block=0.8: total cost = 54,417
  t_block=0.85: total cost = 54,343
  t_block=0.9: total cost = 54,304
  t_block=0.95: total cost = 54,253

Step 2 - MLP REVIEW threshold (holding t_block = 0.95):
  t_review=0.0001: total cost = 54,253, legit txns sent to REVIEW = 53949, fraud missed entirely = 1
  t_review=0.001: total cost = 49,987, legit txns sent to REVIEW = 46900, fraud missed entirely = 2
  t_review=0.01: total cost = 2,438,481, legit txns sent to REVIEW = 28532, fraud missed entirely = 25
  t_review=0.05: total cost = 9,692,901, legit txns sent to REVIEW = 7239, fraud missed entirely = 84
  t_review=0.1: total cost = 16,175,395, legit txns sent to REVIEW = 2166, fraud missed entirely = 114
  t_review=0.2: total cost = 23,096,590, legit txns sent to REVIEW = 549, fraud mi

  t_review=0.4: total cost = 26,651,823, legit txns sent to REVIEW = 188, fraud missed entirely = 146
  t_review=0.5: total cost = 28,104,150, legit txns sent to REVIEW = 129, fraud missed entirely = 150


In [10]:
# Step 3 - narrow in between the neighbours of the best coarse REVIEW threshold.
# The grid is geometric because the candidate thresholds span orders of magnitude, and it always
# includes the best coarse value itself, so narrowing can only match or improve on step 2.
coarse = [r["t_review"] for r in review_sweep]
i_best = int(np.argmin([r["cost"] for r in review_sweep]))
lo, hi = coarse[max(i_best - 1, 0)], coarse[min(i_best + 1, len(coarse) - 1)]
narrow_grid = sorted(set(np.round(np.geomspace(lo, hi, 11), 6).tolist()) | {coarse[i_best]})

print(f"Step 3 - narrowing the REVIEW threshold between {lo} and {hi}:")
narrow_sweep = []
for t_rev in narrow_grid:
    c = total_cost(y_true, p_mlp, amounts, t_block=t_block_step1, t_review=t_rev)
    pct_legit = float(((p_mlp >= t_rev) & (p_mlp < t_block_step1) & (y_true == 0)).sum() / (y_true == 0).sum())
    n_missed = int(((p_mlp < t_rev) & (y_true == 1)).sum())
    narrow_sweep.append({"t_review": t_rev, "cost": c, "legit_flagged": pct_legit, "fraud_missed": n_missed})
    print(f"  t_review={t_rev:g}: cost = {c:,.0f}, legit flagged = {pct_legit:.2%}, fraud missed entirely = {n_missed}")
T_REVIEW_MLP = min(narrow_sweep, key=lambda r: r["cost"])["t_review"]

print(f"\nStep 4 - re-checking the BLOCK threshold at t_review = {T_REVIEW_MLP:g}:")
final_block_sweep = []
for t in BLOCK_GRID:
    if t <= T_REVIEW_MLP:
        continue
    c = total_cost(y_true, p_mlp, amounts, t_block=t, t_review=T_REVIEW_MLP)
    final_block_sweep.append({"t_block": t, "cost": c})
    print(f"  t_block={t}: total cost = {c:,.0f}")
T_BLOCK_MLP = min(final_block_sweep, key=lambda r: r["cost"])["t_block"]
print(f"\nMLP thresholds: T_REVIEW = {T_REVIEW_MLP:g}, T_BLOCK = {T_BLOCK_MLP}")

Step 3 - narrowing the REVIEW threshold between 0.0001 and 0.01:
  t_review=0.0001: cost = 54,253, legit flagged = 4.49%, fraud missed entirely = 1
  t_review=0.000158: cost = 52,404, legit flagged = 4.34%, fraud missed entirely = 1


  t_review=0.000251: cost = 50,938, legit flagged = 4.21%, fraud missed entirely = 1
  t_review=0.000398: cost = 49,675, legit flagged = 4.11%, fraud missed entirely = 1
  t_review=0.000631: cost = 48,551, legit flagged = 4.01%, fraud missed entirely = 1
  t_review=0.001: cost = 49,987, legit flagged = 3.90%, fraud missed entirely = 2
  t_review=0.001585: cost = 117,960, legit flagged = 3.72%, fraud missed entirely = 3
  t_review=0.002512: cost = 386,896, legit flagged = 3.45%, fraud missed entirely = 5
  t_review=0.003981: cost = 579,482, legit flagged = 3.10%, fraud missed entirely = 9
  t_review=0.00631: cost = 1,925,118, legit flagged = 2.73%, fraud missed entirely = 15
  t_review=0.01: cost = 2,438,481, legit flagged = 2.37%, fraud missed entirely = 25

Step 4 - re-checking the BLOCK threshold at t_review = 0.000631:
  t_block=0.5: total cost = 49,036
  t_block=0.6: total cost = 48,912
  t_block=0.7: total cost = 48,793
  t_block=0.75: total cost = 48,748
  t_block=0.8: total cost

  t_block=0.9: total cost = 48,602
  t_block=0.95: total cost = 48,551

MLP thresholds: T_REVIEW = 0.000631, T_BLOCK = 0.95


## All three models at their own thresholds
Each model at the thresholds derived for it, under the same cost function.

In [11]:
thresholds = {
    "LightGBM": {"t_review": 0.0001, "t_block": 0.90},        # 02_modeling.ipynb
    "Random Forest": {"t_review": 0.32, "t_block": 0.85},     # 05_model_enhancement.ipynb
    "MLP": {"t_review": T_REVIEW_MLP, "t_block": T_BLOCK_MLP},
}

operating = {}
for name, t in thresholds.items():
    p = probs[name]
    block = p >= t["t_block"]
    review = (p >= t["t_review"]) & ~block
    allow = ~block & ~review
    operating[name] = {
        **t,
        "total_cost": total_cost(y_true, p, amounts, t["t_block"], t["t_review"]),
        "fraud_missed": int(((y_true == 1) & allow).sum()),
        "amount_missed": float(amounts[(y_true == 1) & allow].sum()),
        "legit_reviewed": int(((y_true == 0) & review).sum()),
        "legit_reviewed_pct": float(((y_true == 0) & review).sum() / (y_true == 0).sum()),
        "legit_blocked": int(((y_true == 0) & block).sum()),
        "block_precision": float(precision_score(y_true, block)),
        "block_recall": float(recall_score(y_true, block)),
        "block_f1": float(f1_score(y_true, block)),
    }

summary = pd.DataFrame(operating).T
summary.insert(0, "pr_auc", pd.Series(pr_auc))
pd.set_option("display.width", 250)
print(summary[["pr_auc", "t_review", "t_block", "total_cost", "fraud_missed", "amount_missed"]].to_string(float_format=lambda v: f"{v:,.4f}"))
print()
print(summary[["legit_reviewed", "legit_reviewed_pct", "legit_blocked", "block_precision", "block_recall", "block_f1"]].to_string(float_format=lambda v: f"{v:,.4f}"))

               pr_auc  t_review  t_block      total_cost  fraud_missed   amount_missed
LightGBM       0.7198    0.0001   0.9000 46,455,418.5200       34.0000 46,402,966.5200
Random Forest  0.8335    0.3200   0.8500     47,547.0000        0.0000          0.0000
MLP            0.8485    0.0006   0.9500     48,550.8000        1.0000         63.8000

               legit_reviewed  legit_reviewed_pct  legit_blocked  block_precision  block_recall  block_f1
LightGBM          51,373.0000              0.0427       186.0000           0.7745        0.7774    0.7760
Random Forest     47,225.0000              0.0393        23.0000           0.9639        0.7482    0.8425
MLP               48,247.0000              0.0401        12.0000           0.9816        0.7798    0.8692


## Amount-stratified recall - MLP
Same quartile bins as the LightGBM and Random Forest tables.

In [12]:
bins = [0, 127091, 441423, 1517771, 10_000_001]
labels = ["Q1 (lowest)", "Q2", "Q3", "Q4 (highest)"]

fraud_view = pd.DataFrame({"amount": amounts, "prob": p_mlp})[y_true == 1]
fraud_view["amount_bucket"] = pd.cut(fraud_view["amount"], bins=bins, labels=labels)
recall_by_bucket_mlp = fraud_view.groupby("amount_bucket", observed=True)["prob"].agg(
    n_fraud="size",
    recall_at_block=lambda p: (p >= T_BLOCK_MLP).mean(),
    recall_at_review=lambda p: (p >= T_REVIEW_MLP).mean(),
)
print(f"MLP (review >= {T_REVIEW_MLP:g}, block >= {T_BLOCK_MLP}):")
print(recall_by_bucket_mlp)

MLP (review >= 0.000631, block >= 0.95):
               n_fraud  recall_at_block  recall_at_review
amount_bucket                                            
Q1 (lowest)        200         0.635000             0.995
Q2                 206         0.640777             1.000
Q3                 190         0.826316             1.000
Q4 (highest)       225         1.000000             1.000


## Blind-spot check
The 37 held-out fraud cases LightGBM scored below 0.1% (notebook 05). Does the MLP flag them?

In [13]:
lgb_misses = (y_true == 1) & (probs["LightGBM"] < 0.001)
n_misses = int(lgb_misses.sum())
mlp_on_misses = p_mlp[lgb_misses]
mlp_flags = int((mlp_on_misses >= T_REVIEW_MLP).sum())
mlp_blocks = int((mlp_on_misses >= T_BLOCK_MLP).sum())
rf_flags = int((probs["Random Forest"][lgb_misses] >= thresholds["Random Forest"]["t_review"]).sum())
print(f"LightGBM catastrophic misses on held_out: {n_misses}")
print(f"  MLP flags (>= its REVIEW threshold):           {mlp_flags} of {n_misses}  ({mlp_blocks} of them at BLOCK)")
print(f"  Random Forest flags (>= its REVIEW threshold): {rf_flags} of {n_misses}")
print("\nMLP probability on these cases:")
print(pd.Series(mlp_on_misses).describe().to_string())

LightGBM catastrophic misses on held_out: 37
  MLP flags (>= its REVIEW threshold):           37 of 37  (14 of them at BLOCK)
  Random Forest flags (>= its REVIEW threshold): 37 of 37

MLP probability on these cases:
count    37.000000
mean      0.499938
std       0.421505
min       0.009981
25%       0.084775
50%       0.361402
75%       0.987420
max       1.000000


## Saving the results
Written to `reports/07_mlp_results.json`. The MLP itself is not saved for deployment - this notebook is a comparison, and the API keeps serving whatever it serves now.

In [14]:
results = {
    "split": "held_out (days 14-16)",
    "n_held_out": int(len(y_true)), "n_fraud_held_out": int(y_true.sum()),
    "mlp_config": {"hidden_layers": list(HIDDEN), "dropout": DROPOUT, "learning_rate": LEARNING_RATE,
                   "weight_decay": WEIGHT_DECAY, "batch_size": BATCH_SIZE, "max_epochs": MAX_EPOCHS,
                   "early_stopping_patience": PATIENCE, "validation_split": "eval_1 (days 8-10)"},
    "pos_weight_sweep": sweep,
    "chosen_pos_weight": best_weight,
    "pr_auc": pr_auc, "ci": ci,
    "mlp_train_pr_auc": mlp_train_pr_auc,
    "mlp_seed_variance": {"seeds": [42, 1, 2, 3, 4], "held_out_pr_auc": seed_scores,
                          "mean": float(np.mean(seed_scores)), "std": float(np.std(seed_scores))},
    "threshold_sweeps": {"block_step1": block_sweep, "review_coarse": review_sweep,
                         "review_narrow": narrow_sweep, "block_final": final_block_sweep},
    "operating_points": operating,
    "mlp_amount_recall": [
        {"quartile": q, "n_fraud": int(r.n_fraud), "recall_at_review": float(r.recall_at_review), "recall_at_block": float(r.recall_at_block)}
        for q, r in recall_by_bucket_mlp.iterrows()
    ],
    "blind_spot": {"lgb_misses": n_misses, "mlp_flags": mlp_flags, "mlp_blocks": mlp_blocks, "rf_flags": rf_flags},
    "n_bootstrap": 200, "seed": SEED,
}
RESULTS_PATH.write_text(json.dumps(results, indent=2))
print("Saved ->", RESULTS_PATH.relative_to(PROJECT_ROOT))

Saved -> reports\07_mlp_results.json


## Findings

All numbers are from the cells above (held-out = days 14-16, 1,202,642 rows, 822 fraud).

| Model | Held-out PR-AUC | 95% CI |
|---|---|---|
| LightGBM (`scale_pos_weight=20`) | 0.7198 | [0.6877, 0.7492] |
| Random Forest (`class_weight={0: 1, 1: 500}`) | 0.8335 | [0.8085, 0.8546] |
| MLP (hidden layers 64-32-16, dropout 0.2, `pos_weight=20`) | 0.8485 | [0.8217, 0.8720] |

**The MLP did not underperform.** The expectation from the literature was that a neural network would do worse than the tree models on this data. It didn't:

- The MLP has the highest held-out PR-AUC point estimate (0.8485). Its CI overlaps Random Forest's, so the MLP and Random Forest are not statistically distinguishable. Its CI does not overlap LightGBM's, so it is clearly ahead of LightGBM as configured in this project.
- It is stable across training seeds: 0.8485 to 0.8526 over five seeds (std 0.0014).
- It is not sensitive to the class weight: eval_1 PR-AUC only moved between 0.8271 and 0.8336 across weights 1 to 500. LightGBM collapsed above weight 20 in notebook 05 and Random Forest needed 500.
- Train vs held-out gap is small (0.8711 vs 0.8485).

**At each model's own thresholds, Random Forest is still slightly cheaper.**

| Model | Review >= | Block >= | Total cost | Fraud allowed | Legitimate sent to review | Legitimate blocked | Block precision | Block recall |
|---|---|---|---|---|---|---|---|---|
| LightGBM | 0.0001 | 0.9 | 46,455,419 | 34 | 4.27% | 186 | 0.7745 | 0.7774 |
| Random Forest | 0.32 | 0.85 | 47,547 | 0 | 3.93% | 23 | 0.9639 | 0.7482 |
| MLP | 0.000631 | 0.95 | 48,551 | 1 | 4.01% | 12 | 0.9816 | 0.7798 |

- Random Forest has the lowest total cost (47,547); the MLP is close behind (48,551). Both are about three orders of magnitude below LightGBM (46,455,419).
- The MLP allows 1 fraud case through (amount 63.80); Random Forest allows 0.
- The MLP's BLOCK decision is the most precise of the three (0.9816, 12 legitimate transactions blocked).
- The MLP flags all 37 of the 37 fraud cases LightGBM scored below 0.1%, as does Random Forest.

**Caveats**
- The MLP's REVIEW threshold (0.000631) sits right next to a cliff: at 0.001 the cost is 49,987 and at 0.001585 it is 117,960. Random Forest's 0.32 has the same property (notebook 05: cost rises from 47,547 at 0.32 to 128,802 at 0.34). Neither threshold has much safety margin.
- As in notebook 05, thresholds were swept on the held-out set itself, so the cost figures are in-sample for the thresholds, for all three models.
- The MLP got one thing the tree models didn't need: a signed-log transform and standardisation of the continuous inputs. That is standard preprocessing for a neural network, not tuning, but it is a difference.
- This compares one configuration of each model. LightGBM in particular is the original `scale_pos_weight=20` model, which has a known blind spot; the result says the MLP beats *that* LightGBM, not gradient boosting in general.
- PaySim is synthetic and its fraud follows simple generated rules, which may be easier for a small network to learn than real fraud. The same network was retrained on the ULB data in `06_cross_dataset_validation.ipynb`: held-out PR-AUC 0.7972 (95% CI [0.7312, 0.8613]), the highest point estimate there, with a confidence interval that overlaps the logistic regression baseline and LightGBM.
- The MLP is not deployed. The API keeps serving the Random Forest.